In [ ]:
from pathlib import Path
import pandas as pd

import teehr
from teehr import RemoteReadWriteEvaluation
from teehr.utilities.apply_migrations import evolve_catalog_schema

from setup_utils import create_minio_spark_session, DEV_LOCATION_ID_LIST

### Create spark and init eval

In [ ]:
spark = create_minio_spark_session()

In [ ]:
migrations_dir = Path(teehr.__file__).parent / "migrations"

evolve_catalog_schema(
    spark=spark,
    migrations_dir_path=migrations_dir,
    target_catalog_name="iceberg",
    target_namespace_name="teehr"
)

In [ ]:
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

### Load JTS to the warehouse

In [ ]:
inputs_dir = Path(Path.cwd(), 'FIRO_data')

# load joined_timeseries from parquet
joined_timeseries_path = Path(inputs_dir, 'joined_timeseries.parquet')
sdf = spark.read.parquet(joined_timeseries_path)

In [ ]:
ev._write.to_warehouse(
    source_data=sdf,
    table_name='joined_timeseries',
    write_mode="create_or_replace"
)

### Kill spark

In [ ]:
spark.stop()